In [ ]:
import sys, os
# Adjust depth based on notebook location relative to project root
_src_path = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from project_config import load_config

config    = load_config()
base_dir  = config['base_dir']
well_list = config['well_list']
well_meta = config.get('well_metadata', {})

print(f'Active experiment: {config.get("experiment_name", config.get("experiment_key", "?"))}')
print(f'Base dir: {base_dir}')

In [1]:
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn import preprocessing
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
import phate
import math
import random
import gc
import scprep
from datetime import datetime, time
from matplotlib.animation import ImageMagickWriter
import matplotlib.animation as animation
import zipfile
from urllib.request import urlopen
import scipy.stats as st
from scipy.stats import norm
from scipy.stats import gaussian_kde
from scipy.stats import kde
from scipy.stats import binned_statistic
from scipy.stats import f_oneway
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['pdf.fonttype'] = 42
print(sns.__version__)
from anndata import AnnData
import scanpy as sc
from delve import *
import anndata as ad
from sklearn.preprocessing import MinMaxScaler
from kh import sketch
from sklearn.cluster import KMeans
import umap
print(sc.__version__)
today = datetime.now().strftime("%m%d%Y-%H%M")

0.11.2
1.9.1


In [2]:
# Read back in the subsampled adata file
adata_save_path = os.path.join(base_dir, 'standard_adata_sub.h5ad')
standard_adata_sub = ad.read_h5ad(adata_save_path)

# 🔹 Filter only specific sample IDs right after loading

#LPS 246 Alpelisib + Abema
#All 
selected_sample_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
#0nM Abema
#selected_sample_ids = [1, 2, 3, 4, 5, 6, 7]
#10nM Abema
#selected_sample_ids = [1, 8, 9, 10, 11, 12, 13, 14]
#50nM Abema
#selected_sample_ids = [1, 15, 16, 17, 18, 19, 20, 21]
standard_adata_sub = standard_adata_sub[
    standard_adata_sub.obs['sample_ID'].isin(selected_sample_ids)
].copy()


In [3]:
standard_adata_sub.obs

,label,well,sample_ID,treatment,dose,centroid-0,centroid-1,pRB_label,pRb_Rb_ratio
C_71026,5025,C02,1.0,Alp,0uM,3093.247692,5219.136923,pRB_high,0.928798
C_8192,8238,B02,1.0,Alp,0uM,4933.103627,5699.663212,pRB_high,1.274556
C_8210,8256,B02,1.0,Alp,0uM,4938.099217,6544.469974,pRB_high,1.910379
C_8211,8257,B02,1.0,Alp,0uM,4941.076350,3081.324022,pRB_high,0.991877
C_8214,8260,B02,1.0,Alp,0uM,4938.290393,5342.519651,pRB_high,1.546198
...,...,...,...,...,...,...,...,...,...
C_423279,7483,G08,21.0,Alp + Abema,20uM + 50nM,5822.105860,1586.879017,pRB_low,-1.298132
C_423284,7488,G08,21.0,Alp + Abema,20uM + 50nM,5820.963124,1028.722343,pRB_high,-0.272877
C_351088,8259,F08,21.0,Alp + Abema,20uM + 50nM,6226.011765,1555.776471,pRB_high,0.062952
C_351089,8260,F08,21.0,Alp + Abema,20uM + 50nM,6240.505435,5392.029891,pRB_high,0.373262


In [4]:
def laplacian_score_fs(adata = None,
                    k: int  = None,
                    n_jobs: int  = -1):

    X, feature_names, obs_names = parse_input(adata)
    W = construct_affinity(X = X, k = k, n_jobs = n_jobs)
    scores = laplacian_score(X = X, W = W)
    predicted_features = pd.DataFrame(scores, index = feature_names, columns = ['laplacian_score'])
    predicted_features = predicted_features.sort_values(by = 'laplacian_score', ascending = True)

    return predicted_features 

In [5]:
#l_score_fullest = laplacian_score_fs(adata_fullest_sub, k = 200)
l_score_standard = laplacian_score_fs(standard_adata_sub, k = 100)
#l_score_normalized = laplacian_score_fs(normalized_trimmed_noPSTAT5_adata_sub, k = 200)

In [6]:
len(l_score_standard)

36

In [7]:
l_score_standard

,laplacian_score
R0_pRb_total_nuc_protein,0.059451
R1_CDK2_total_nuc_protein,0.064247
R0_pRb_nuc_cyto_ratio,0.067850
R0_pRb_nuc_mean,0.068791
R0_Rb_total_nuc_protein,0.073127
R0_DNA_total_nuc_protein,0.075308
total_DNA,0.075308
R1_DNA_total_nuc_protein,0.076832
R1_CDK2_nuc_mean,0.078556
R1_pS6_nuc_mean,0.084454


Calculating PHATE Plot Coordinates

In [11]:
import os
import numpy as np
import phate
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pptx import Presentation
from pptx.util import Inches
from io import BytesIO
import pandas as pd

# Define parameter ranges
count_list = [80, 100, 120, 140]  # nearest neighbors
index_ranges = [10, 15, 20, 30, 35]  # how many features it is considering
t_values = ['auto']
gamma_values = [1]

# Create a PowerPoint presentation object
presentation = Presentation()

# Loop over each combination of parameters
for count in count_list:
    for index_range in index_ranges:
        for t in t_values:
            for gamma in gamma_values:
                # Initialize PHATE operator with varying t and gamma, projecting to three dimensions
                phate_op = phate.PHATE(knn=count, t=t, gamma=gamma, n_components=3)

                # Fit-transform PHATE
                X_phate = phate_op.fit_transform(
                    standard_adata_sub.X[
                        :, np.isin(
                            standard_adata_sub.var_names,
                            l_score_standard.index[:index_range]
                        )
                    ]
                )

                # Set PHATE result for plotting
                standard_adata_sub.obsm['X_phate'] = X_phate

                # Generate plot title
                plot_title = f'PHATE: Neighbors={count}, Index Range={index_range}, t={t}, gamma={gamma}'

                # Define angles for rotation in degrees
                angles = [(30, 30), (30, 120), (30, 210), (30, 300)]

                # ----- Dose labels sorted numerically (handles strings like "10 nM") -----
                dose_series = standard_adata_sub.obs['dose'].astype(str).str.strip()
                tmp = pd.DataFrame({'label': dose_series.unique()})
                tmp['num'] = tmp['label'].str.extract(r'([\d.]+)').astype(float)
                ordered_labels = tmp.sort_values('num', na_position='last')['label'].tolist()

                # Build colormap in numeric order
                cmap = plt.get_cmap('viridis', len(ordered_labels))
                colors = cmap(np.linspace(0, 1, len(ordered_labels)))
                label_to_color = dict(zip(ordered_labels, colors))
                # ------------------------------------------------------------------------

                # Create a new slide for the current set of plots
                slide = presentation.slides.add_slide(presentation.slide_layouts[5])
                title = slide.shapes.title
                title.text = plot_title

                for i, (elev, azim) in enumerate(angles, start=1):
                    fig = plt.figure()
                    ax = fig.add_subplot(111, projection='3d')

                    # Plot each label in numeric order with a unique color
                    handles = []
                    for label in ordered_labels:
                        color = label_to_color[label]
                        idx = (standard_adata_sub.obs['dose'].astype(str).str.strip() == label)
                        h = ax.scatter(
                            X_phate[idx, 0], X_phate[idx, 1], X_phate[idx, 2],
                            color=color, label=label, s=5
                        )
                        handles.append(h)

                    ax.view_init(elev=elev, azim=azim)
                    ax.set_title(plot_title)
                    ax.legend(handles=handles, title='dose', bbox_to_anchor=(1.05, 1), loc='upper left')

                    # Save the plot to a BytesIO object
                    img_stream = BytesIO()
                    plt.savefig(img_stream, format='png', bbox_inches='tight')
                    plt.close()
                    img_stream.seek(0)

                    # Add the plot to the current slide in a 2x2 grid
                    left = Inches(1 + ((i - 1) % 2) * 4)
                    top = Inches(1.5 + ((i - 1) // 2) * 3.5)
                    slide.shapes.add_picture(img_stream, left, top, width=Inches(3.5))

# Save the PowerPoint presentation
pptx_filename = 'phate_plots_presentation.pptx'
presentation.save(pptx_filename)

print(f"3D PHATE plots saved with multiple angles and legends. PowerPoint presentation saved as {pptx_filename}.")


Calculating PHATE...
  Running PHATE on 42000 observations and 10 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculated KNN search in 22.56 seconds.
    Calculating affinities...
    Calculated affinities in 1.54 seconds.
  Calculated graph and diffusion operator in 24.38 seconds.
  Calculating landmark operator...
    Calculating SVD...
    Calculated SVD in 10.16 seconds.
    Calculating KMeans...
    Calculated KMeans in 5.69 seconds.
  Calculated landmark operator in 16.97 seconds.
  Calculating optimal t...
    Automatically selected t = 16
  Calculated optimal t in 3.85 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.20 seconds.
  Calculating metric MDS...
  Calculated metric MDS in 24.66 seconds.
Calculated PHATE in 70.08 seconds.
Calculating PHATE...
  Running PHATE on 42000 observations and 15 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculated 

For plotting different data onto 3D PHATE structures

In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
from pptx import Presentation
from pptx.util import Inches, Cm
from io import BytesIO

# Define angles for rotation in degrees
angles = [(30, 30), (30, 120), (30, 210), (30, 300)]

# Ensure the directory for saving plots exists
plots_dir = 'phate_plots'
os.makedirs(plots_dir, exist_ok=True)

# Create a PowerPoint presentation object
prs = Presentation()
slide_width = prs.slide_width
slide_height = prs.slide_height

# Get all features in .var_names
features = standard_adata_sub.var_names

# Loop over each feature
for feature in features:
    # Obtain expression data for the feature
    data = standard_adata_sub[:, feature].X

    # Cap high and low values for visualization
    percentile_99 = np.percentile(data, 99)
    percentile_1 = np.percentile(data, 1)
    data_capped = np.clip(data, percentile_1, percentile_99)

    # Normalize and map data to colors
    norm = plt.Normalize(np.min(data_capped), np.max(data_capped))
    cmap = plt.get_cmap('viridis')
    colors = cmap(norm(data_capped))

    # Create a single figure with a 2x2 grid of subplots
    fig = plt.figure(figsize=(16, 16))
    for i, (elev, azim) in enumerate(angles, start=1):
        ax = fig.add_subplot(2, 2, i, projection='3d')
        # Plot the feature data
        ax.scatter(standard_adata_sub.obsm['X_phate'][:, 0], 
                   standard_adata_sub.obsm['X_phate'][:, 1], 
                   standard_adata_sub.obsm['X_phate'][:, 2], 
                   color=colors, label=feature, s=5)
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(f'View Angle {i}: {feature}')

    # Save plot to a BytesIO object
    canvas = FigureCanvas(fig)
    canvas.draw()
    image_stream = BytesIO()
    plt.savefig(image_stream, format='png')
    plt.close(fig)

    # Add a slide to the presentation and remove default placeholders
    slide_layout = prs.slide_layouts[5]  # Using a blank slide layout
    slide = prs.slides.add_slide(slide_layout)
    for placeholder in slide.placeholders:
        placeholder.element.getparent().remove(placeholder.element)
    
    # Calculate image size to maximize slide use while keeping aspect ratio
    aspect_ratio = fig.get_figwidth() / fig.get_figheight()
    slide_aspect_ratio = slide_width / slide_height
    if aspect_ratio > slide_aspect_ratio:
        width = slide_width
        height = slide_width / aspect_ratio
    else:
        height = slide_height
        width = slide_height * aspect_ratio

    left = (slide_width - width) / 2
    top = (slide_height - height) / 2

    pic = slide.shapes.add_picture(image_stream, left, top, width=width, height=height)
    image_stream.close()

# Save the presentation
prs.save(f'{plots_dir}/phate_plots_presentation.pptx')
print("PowerPoint presentation saved with combined 3D PHATE plots optimized to fill slides.")


PowerPoint presentation saved with combined 3D PHATE plots optimized to fill slides.


After you've decided on a final structure, you can save the ADATA file. It should have the PHATE coordinates saved as part of it now, removing the need to recompute the phate structure if you want to plot different things onto it later

In [13]:
#Save the entire adata file with new PHATE embeddings
adata_save_path = os.path.join(base_dir, 'standard_adata_sub_PHATE.h5ad')
standard_adata_sub.write_h5ad(adata_save_path)